In [1]:
# ============================================================
# Cell 1: ENVIRONMENT GATE — Colab A100 ONLY
# ============================================================
# MANDATORY: This cell must pass before ANY training.
# Refuses local execution. Verifies CUDA GPU.
import os, sys

_cwd = os.getcwd()
_is_local = _cwd.startswith("/Users/") or (_cwd.startswith("/home/") and "content" not in _cwd)

import torch
_has_cuda = torch.cuda.is_available()

if _is_local or not _has_cuda:
    print("=" * 65)
    print("  BLOCKED: This notebook must run on Google Colab with CUDA GPU")
    print(f"  Current dir : {_cwd}")
    print(f"  CUDA        : {_has_cuda}")
    print("=" * 65)
    print("\n  Runtime → Change runtime type → A100 GPU")
    raise SystemExit("Refusing local execution. Use Colab A100.")

# Passed — CUDA environment verified
gpu_name = torch.cuda.get_device_name(0)
props = torch.cuda.get_device_properties(0)
vram_gb = props.total_memory / 1e9
cc = (props.major, props.minor)

print(f"{'='*65}")
print(f"  ENVIRONMENT VERIFIED")
print(f"  GPU          : {gpu_name}")
print(f"  VRAM         : {vram_gb:.1f} GB")
print(f"  Compute cap  : {cc[0]}.{cc[1]}")
print(f"  PyTorch      : {torch.__version__}")
print(f"  CUDA         : {torch.version.cuda}")
print(f"  Working dir  : {_cwd}")
print(f"{'='*65}")
os.system("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader")

  ENVIRONMENT VERIFIED
  GPU          : NVIDIA A100-SXM4-40GB
  VRAM         : 42.4 GB
  Compute cap  : 8.0
  PyTorch      : 2.10.0+cu128
  CUDA         : 12.8
  Working dir  : /content


0

In [118]:
import subprocess, os
os.chdir("/content/nst")
r = subprocess.run(["git", "pull", "--ff-only"], capture_output=True, text=True)
print(r.stdout, r.stderr)
# Verify the fix
r2 = subprocess.run(["grep", "min_evidence_words", "training/train_fever_veri.py"], capture_output=True, text=True)
print("Verify fix:", r2.stdout)

Updating dbf9be0..db4f0a2
Fast-forward
 models/nst_veri.py | 5 ++++-
 1 file changed, 4 insertions(+), 1 deletion(-)
 From https://github.com/poolanithinreddy/Neurosymbolic-Transformers
   dbf9be0..db4f0a2  main       -> origin/main

Verify fix:                     constraint_signals = constraint_engine.evaluate_batch(claims, evidences, min_evidence_words=2)



In [ ]:
import subprocess, sys, json, os, shutil, time, re

seeds = [42, 123, 456]
results = {"neural": [], "veri": []}

commands = {
    "neural": ("train-fever-nst", "fever_neural_10k_a100"),
    "veri": ("train-fever-veri", "fever_veri_10k_a100"),
}

for seed in seeds:
    for mode in ["neural", "veri"]:
        subcmd, cfg_name = commands[mode]
        outdir = f"/content/nst/outputs_{mode}_10k"
        cfg_path = f"/content/nst/configs/{cfg_name}.yaml"
        
        # Clean previous
        if os.path.isdir(outdir):
            shutil.rmtree(outdir)
        
        # Patch seed
        with open(cfg_path) as f:
            cfg_text = f.read()
        patched = re.sub(r'seed:\s*\d+', f'seed: {seed}', cfg_text)
        with open(cfg_path, 'w') as f:
            f.write(patched)
        
        print(f"\n{'='*50}")
        print(f"  {mode.upper()} seed={seed}")
        print(f"{'='*50}")
        
        t0 = time.time()
        ret = subprocess.run(
            [sys.executable, "main.py", subcmd,
             "--config", f"configs/{cfg_name}.yaml",
             "--outdir", f"outputs_{mode}_10k"],
            cwd="/content/nst"
        )
        elapsed = time.time() - t0
        
        # Restore config
        with open(cfg_path, 'w') as f:
            f.write(cfg_text)
        
        # Read results
        report_path = f"{outdir}/report.json"
        if os.path.exists(report_path):
            with open(report_path) as f:
                res = json.load(f)
            dev_acc = res.get("dev", {}).get("accuracy", res.get("best_dev_acc", 0))
            held = res.get("dev_test", {}).get("accuracy", 0)
            results[mode].append({"seed": seed, "dev": dev_acc, "held": held, "time": elapsed})
            print(f"  ✓ dev={dev_acc:.4f} held={held:.4f} ({elapsed/60:.1f}m)")
        else:
            print(f"  ✗ FAILED (code={ret.returncode}, {elapsed/60:.1f}m)")
            results[mode].append({"seed": seed, "dev": 0, "held": 0, "time": elapsed})

# Summary
import statistics
print(f"\n{'='*60}")
print("MULTI-SEED COMPARISON (3 seeds)")
print(f"{'='*60}")
for mode in ["neural", "veri"]:
    valid = [r for r in results[mode] if r["dev"] > 0]
    if len(valid) >= 2:
        devs = [r["dev"] for r in valid]
        helds = [r["held"] for r in valid]
        print(f"{mode}: dev={statistics.mean(devs):.4f}±{statistics.stdev(devs):.4f}  held={statistics.mean(helds):.4f}±{statistics.stdev(helds):.4f}")
    for r in valid:
        print(f"  seed={r['seed']}: dev={r['dev']:.4f} held={r['held']:.4f}")

n_valid = [r for r in results["neural"] if r["dev"] > 0]
v_valid = [r for r in results["veri"] if r["dev"] > 0]
if len(n_valid) >= 2 and len(v_valid) >= 2:
    n_dev = statistics.mean([r["dev"] for r in n_valid])
    v_dev = statistics.mean([r["dev"] for r in v_valid])
    n_held = statistics.mean([r["held"] for r in n_valid])
    v_held = statistics.mean([r["held"] for r in v_valid])
    print(f"\nMean Delta: dev={v_dev-n_dev:+.4f}  held={v_held-n_held:+.4f}")
    
    # Win count
    dev_wins = sum(1 for v, n in zip(v_valid, n_valid) if v["dev"] > n["dev"])
    held_wins = sum(1 for v, n in zip(v_valid, n_valid) if v["held"] > n["held"])
    print(f"VERI wins: {dev_wins}/{len(v_valid)} on dev, {held_wins}/{len(v_valid)} on held-out")

STDOUT: (empty)
STDERR: (empty)
Return code: 0


In [ ]:
# ============================================================
# Cell 1: ENVIRONMENT GATE — Colab A100 ONLY
# ============================================================
# MANDATORY: This cell must pass before ANY training.
# Refuses local execution. Verifies CUDA GPU.
import os, sys

_cwd = os.getcwd()
_is_local = _cwd.startswith("/Users/") or (_cwd.startswith("/home/") and "content" not in _cwd)

import torch
_has_cuda = torch.cuda.is_available()

if _is_local or not _has_cuda:
    print("=" * 65)
    print("  BLOCKED: This notebook must run on Google Colab with CUDA GPU")
    print(f"  Current dir : {_cwd}")
    print(f"  CUDA        : {_has_cuda}")
    print("=" * 65)
    print("\n  Runtime → Change runtime type → A100 GPU")
    raise SystemExit("Refusing local execution. Use Colab A100.")

# Passed — CUDA environment verified
gpu_name = torch.cuda.get_device_name(0)
props = torch.cuda.get_device_properties(0)
vram_gb = props.total_memory / 1e9
cc = (props.major, props.minor)

print(f"{'='*65}")
print(f"  ENVIRONMENT VERIFIED")
print(f"  GPU          : {gpu_name}")
print(f"  VRAM         : {vram_gb:.1f} GB")
print(f"  Compute cap  : {cc[0]}.{cc[1]}")
print(f"  PyTorch      : {torch.__version__}")
print(f"  CUDA         : {torch.version.cuda}")
print(f"  Working dir  : {_cwd}")
print(f"{'='*65}")
os.system("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader")
# ============================================================
# Cell 1: ENVIRONMENT GATE — Colab A100 ONLY
# ============================================================
# MANDATORY: This cell must pass before ANY training.
# Refuses local execution. Verifies CUDA GPU.
import os, sys

_cwd = os.getcwd()
_is_local = _cwd.startswith("/Users/") or (_cwd.startswith("/home/") and "content" not in _cwd)

import torch
_has_cuda = torch.cuda.is_available()

if _is_local or not _has_cuda:
    print("=" * 65)
    print("  BLOCKED: This notebook must run on Google Colab with CUDA GPU")
    print(f"  Current dir : {_cwd}")
    print(f"  CUDA        : {_has_cuda}")
    print("=" * 65)
    print("\n  Runtime → Change runtime type → A100 GPU")
    raise SystemExit("Refusing local execution. Use Colab A100.")

# Passed — CUDA environment verified
gpu_name = torch.cuda.get_device_name(0)
props = torch.cuda.get_device_properties(0)
vram_gb = props.total_memory / 1e9
cc = (props.major, props.minor)

print(f"{'='*65}")
print(f"  ENVIRONMENT VERIFIED")
print(f"  GPU          : {gpu_name}")
print(f"  VRAM         : {vram_gb:.1f} GB")
print(f"  Compute cap  : {cc[0]}.{cc[1]}")
print(f"  PyTorch      : {torch.__version__}")
print(f"  CUDA         : {torch.version.cuda}")
print(f"  Working dir  : {_cwd}")
print(f"{'='*65}")
os.system("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader")
# ============================================================
# Cell 2: Clone Repo + Install Dependencies
# ============================================================
import subprocess, os, sys, pathlib

REPO_URL = "https://github.com/poolanithinreddy/Neurosymbolic-Transformers.git"
PROJ_ROOT = "/content/nst"

if not os.path.isdir(os.path.join(PROJ_ROOT, "data")):
    print("Cloning repository...")
    subprocess.run(["git", "clone", REPO_URL, PROJ_ROOT], check=True)
else:
    print("Repository already cloned. Pulling latest...")
    subprocess.run(["git", "-C", PROJ_ROOT, "pull", "--ff-only"], check=True)

os.chdir(PROJ_ROOT)
if PROJ_ROOT not in sys.path:
    sys.path.insert(0, PROJ_ROOT)

# Install package + deps
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", PROJ_ROOT], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "datasets==2.21.0", "peft>=0.9", "transformers>=4.40",
                "accelerate", "sentencepiece", "protobuf"], check=True)

print(f"\nProject root: {PROJ_ROOT}")
print(f"Python: {sys.executable}")

# GPU auto-config
import torch
props = torch.cuda.get_device_properties(0)
vram_gb = props.total_memory / 1e9
cc = (props.major, props.minor)
supports_bf16 = cc >= (8, 0)

if supports_bf16:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True

# Determine batch config based on VRAM
if vram_gb >= 35:
    BS, GA = 32, 2
elif vram_gb >= 20:
    BS, GA = 24, 2
else:
    BS, GA = 16, 2

GPU_OVERRIDES = {
    "train": {
        "batch_size": BS,
        "grad_accum_steps": GA,
        "bf16": supports_bf16,
        "fp16": not supports_bf16,
        "tf32": supports_bf16,
        "fused_optimizer": True,
        "num_workers": 4,
    }
}

DEVICE = "cuda"
prec = "BF16" if supports_bf16 else "FP16"
print(f"\nGPU config: bs={BS}×{GA}={BS*GA} effective, {prec}, VRAM={vram_gb:.0f}GB")
# ============================================================
# Cell 3: Verify datasets version compatibility
# ============================================================
import datasets
print(f"datasets version: {datasets.__version__}")
_ds_major = int(datasets.__version__.split(".")[0])
assert _ds_major < 3, (
    f"datasets {datasets.__version__} doesn't support FEVER loading scripts. "
    "Run: pip install datasets==2.21.0 then restart the kernel."
)
print("OK — compatible with FEVER loading script")
# ============================================================
# Cell 4: Build Wiki Cache & Verify Evidence Quality
# ============================================================
# The FEVER dataset needs a wiki page cache to resolve evidence
# text from page title + sentence index. Without it, evidence is
# just page titles → NLI accuracy capped at ~60-70%.
import logging, os, time, sys
logging.basicConfig(level=logging.INFO, format="%(name)s | %(message)s", force=True)

# Clear stale project modules for fresh imports
for mod in list(sys.modules.keys()):
    if any(mod.startswith(p) for p in ["data.", "models.", "training.", "eval.",
                                        "logic.", "symbolic.", "retrieval."]):
        del sys.modules[mod]

# ── Step 1: Build wiki cache if missing ──
from data.fever_wiki_cache import build_wiki_cache, cache_stats

cache_path = os.path.join(PROJ_ROOT, "data", "fever_wiki.db")
stats = cache_stats(cache_path)
if stats.get("exists"):
    print(f"Wiki cache exists: {stats['n_pages']} pages, {stats['size_mb']:.1f} MB")
else:
    print("Building wiki page cache (one-time, ~5-10 min)...")
    t0 = time.time()
    build_stats = build_wiki_cache(cache_path=cache_path)
    elapsed = time.time() - t0
    print(f"Done in {elapsed:.0f}s: {build_stats['n_found']}/{build_stats['n_needed']} pages")

# ── Step 2: Load small data sample and verify evidence quality ──
from data.fever_dataset import load_fever_splits, print_fever_stats

splits_check = load_fever_splits(max_train=500, max_dev=200, dev_test_ratio=0.1, seed=42)
print_fever_stats(splits_check)

train_items = splits_check["train"]
n_with_evidence = sum(1 for it in train_items if len(it.get("gold_evidence_text", "")) > 30)
pct = 100 * n_with_evidence / max(1, len(train_items))
print(f"\nEvidence quality: {n_with_evidence}/{len(train_items)} ({pct:.0f}%) have >30 char evidence")

# Show sample evidence for manual verification
for i in range(min(5, len(train_items))):
    it = train_items[i]
    ev = it.get("gold_evidence_text", "")[:150]
    print(f"\n  [{i}] {it['label']}: {it['claim'][:80]}")
    print(f"       Evidence: {ev}")

if pct > 60:
    print(f"\n{'='*50}")
    print(f"  EVIDENCE CHECK PASSED — {pct:.0f}% coverage")
    print(f"{'='*50}")
elif pct > 20:
    print(f"\n  WARNING: Partial evidence ({pct:.0f}%). Results will be degraded.")
else:
    print(f"\n  CRITICAL: Only {pct:.0f}% have evidence. Wiki cache needed.")
    print(f"  Run: python main.py build-fever-wiki-cache")
# ============================================================
# Cell 5: Smoke Test — 50 examples (validates pipeline end-to-end)
# ============================================================
import time, gc, json, sys

for mod in list(sys.modules.keys()):
    if any(mod.startswith(p) for p in ["data.", "models.", "training.", "eval.",
                                        "logic.", "symbolic.", "retrieval."]):
        del sys.modules[mod]
gc.collect()
torch.cuda.empty_cache()

print("Smoke test: 50 train / 25 dev / 1 epoch (VERI mode)")
print("Purpose: verify pipeline runs end-to-end on CUDA\n")

t0 = time.time()
from training.train_fever_veri import train_fever_veri

results_smoke = train_fever_veri(
    "configs/fever_veri_3k_a100.yaml",
    config_overrides={
        "data": {"max_train": 50, "max_dev": 25, "dev_sample": 25},
        "train": {"epochs": 1, "eval_every_steps": 25, "patience": 100},
        "io": {"out_dir": "outputs_smoke_veri"},
    }
)
elapsed = time.time() - t0

dev = results_smoke.get("dev", {})
print(f"\nSmoke test complete in {elapsed:.1f}s")
print(f"  Dev acc: {dev.get('accuracy', 'N/A')}")
print(f"  (Accuracy is meaningless at 50 examples — this just validates the pipeline)")

# Verify CUDA was actually used
assert DEVICE == "cuda", "ERROR: Not running on CUDA!"
print(f"\n  Pipeline smoke test PASSED on {torch.cuda.get_device_name(0)}")
# ============================================================
# Cell 6: 3K NEURAL BASELINE (Fair Comparison — Same Architecture)
# ============================================================
# DeBERTa-v3-large + LoRA, NO constraints.
# Establishes the ceiling that pure neural achieves on 3k.
import time, json, gc, sys

for mod in list(sys.modules.keys()):
    if any(mod.startswith(p) for p in ["data.", "models.", "training.", "eval.",
                                        "logic.", "symbolic.", "retrieval."]):
        del sys.modules[mod]
gc.collect()
torch.cuda.empty_cache()

print("=" * 65)
print("  3K NEURAL BASELINE: DeBERTa-v3-large + LoRA (NO constraints)")
print("=" * 65)

t0 = time.time()
from training.train_fever_nst import train_fever_nst
results_neural_3k = train_fever_nst(
    "configs/fever_neural_3k_a100.yaml",
    config_overrides=GPU_OVERRIDES,
)
elapsed = time.time() - t0

dev = results_neural_3k.get("dev", {})
print(f"\n{'='*65}")
print(f"  NEURAL BASELINE 3K RESULTS ({elapsed/60:.1f} min)")
print(f"{'='*65}")
print(f"  Dev acc    : {dev.get('accuracy', 'N/A')}")
print(f"  Dev ECE    : {dev.get('ece', 'N/A')}")
print(f"  Best dev   : {results_neural_3k.get('best_dev_acc', 'N/A')}")
for label, stats in dev.get("per_label", {}).items():
    print(f"    {label:<20}: {stats.get('accuracy', 0):.4f} (n={stats.get('count', 0)})")

with open("results_neural_3k.json", "w") as f:
    json.dump(results_neural_3k, f, indent=2, default=str)
print(f"\n  Saved to results_neural_3k.json")
# ============================================================
# Cell 7: 3K NST-VERI (Constraint-Enhanced Training — THE CRITICAL RUN)
# ============================================================
# DeBERTa-v3-large + LoRA + verification heads + contrastive + adaptive lambda
# 3-phase: NLI+aux → +contrastive → +constraints
#
# WATCH FOR:
#   - constraint_loss > 0 (constraints must be active)
#   - fire_rate > 0 (constraints must fire)
#   - mean_lambda > 0.1 (constraints must have weight)
#   - dev_acc >= neural baseline (constraints must not hurt)
import time, json, gc, sys

for mod in list(sys.modules.keys()):
    if any(mod.startswith(p) for p in ["data.", "models.", "training.", "eval.",
                                        "logic.", "symbolic.", "retrieval."]):
        del sys.modules[mod]
gc.collect()
torch.cuda.empty_cache()

print("=" * 65)
print("  3K NST-VERI: Verification-Enhanced Constraint Training")
print("=" * 65)

t0 = time.time()
from training.train_fever_veri import train_fever_veri
results_veri_3k = train_fever_veri(
    "configs/fever_veri_3k_a100.yaml",
    config_overrides=GPU_OVERRIDES,
)
elapsed = time.time() - t0

dev = results_veri_3k.get("dev", {})
print(f"\n{'='*65}")
print(f"  NST-VERI 3K RESULTS ({elapsed/60:.1f} min)")
print(f"{'='*65}")
print(f"  Dev acc    : {dev.get('accuracy', 'N/A')}")
print(f"  Dev ECE    : {dev.get('ece', 'N/A')}")
print(f"  Best dev   : {results_veri_3k.get('best_dev_acc', 'N/A')}")
for label, stats in dev.get("per_label", {}).items():
    print(f"    {label:<20}: {stats.get('accuracy', 0):.4f} (n={stats.get('count', 0)})")

# ── Constraint activity analysis ──
train_log = results_veri_3k.get("train_log", [])
if train_log:
    phase3_entries = [e for e in train_log if e.get("phase", 0) >= 3]
    if phase3_entries:
        cst_losses = [e.get("loss_constraint", 0) for e in phase3_entries]
        lambdas = [e.get("mean_lambda", 0) for e in phase3_entries]
        print(f"\n  CONSTRAINT DIAGNOSTICS:")
        print(f"    Phase 3 entries   : {len(phase3_entries)}")
        print(f"    Constraint loss   : min={min(cst_losses):.4f} max={max(cst_losses):.4f} mean={sum(cst_losses)/len(cst_losses):.4f}")
        print(f"    Mean lambda       : min={min(lambdas):.4f} max={max(lambdas):.4f} mean={sum(lambdas)/len(lambdas):.4f}")
        if max(cst_losses) > 0.001:
            print(f"    CONSTRAINTS ARE ACTIVE")
        else:
            print(f"    WARNING: CONSTRAINTS STILL INACTIVE")
    else:
        print(f"\n  WARNING: No Phase 3 entries in training log")

# Constraint calibration
calib = results_veri_3k.get("constraint_calibration", {})
if calib:
    print(f"\n  CONSTRAINT CALIBRATION (on dev):")
    for cname, cstats in calib.items():
        print(f"    {cname}: precision={cstats.get('precision', 0):.3f} fire_rate={cstats.get('fire_rate', 0):.3f}")

with open("results_veri_3k.json", "w") as f:
    json.dump(results_veri_3k, f, indent=2, default=str)
print(f"\n  Saved to results_veri_3k.json")
# ============================================================
# Cell 8: 3K COMPARISON — Neural vs NST-VERI
# ============================================================
import json, os

experiments = {}
for name, path in [("neural_3k", "results_neural_3k.json"),
                   ("veri_3k", "results_veri_3k.json")]:
    if os.path.exists(path):
        with open(path) as f:
            experiments[name] = json.load(f)

print(f"{'='*65}")
print(f"  3K COMPARISON: Neural vs NST-VERI")
print(f"{'='*65}")
print(f"  {'Method':<25} {'Dev Acc':>10} {'Best Acc':>10} {'ECE':>8}")
print(f"  {'─'*55}")

for name, r in experiments.items():
    dev = r.get("dev", {})
    acc = dev.get("accuracy", "N/A")
    best = r.get("best_dev_acc", "N/A")
    ece = dev.get("ece", "N/A")
    acc_s = f"{acc:.4f}" if isinstance(acc, (int, float)) else str(acc)
    best_s = f"{best:.4f}" if isinstance(best, (int, float)) else str(best)
    ece_s = f"{ece:.4f}" if isinstance(ece, (int, float)) else str(ece)
    print(f"  {name:<25} {acc_s:>10} {best_s:>10} {ece_s:>8}")

# ── Signal assessment ──
if "neural_3k" in experiments and "veri_3k" in experiments:
    n_acc = experiments["neural_3k"].get("best_dev_acc", 0)
    v_acc = experiments["veri_3k"].get("best_dev_acc", 0)
    delta = v_acc - n_acc
    print(f"\n  Delta (VERI - Neural): {delta:+.4f}")
    if delta > 0.01:
        print(f"  SIGNAL: NST-VERI shows improvement. Full run justified.")
    elif delta > -0.01:
        print(f"  NEUTRAL: No clear signal yet. May need config tuning.")
    else:
        print(f"  WARNING: NST-VERI underperforms. Investigate before full run.")
# ============================================================
# DIAGNOSTIC: Why do SUPPORTS/REFUTES examples lack evidence?
# ============================================================
import json, sys, os
for mod in list(sys.modules.keys()):
    if any(mod.startswith(p) for p in ["data.", "models.", "training.", "eval.",
                                        "logic.", "symbolic.", "retrieval."]):
        del sys.modules[mod]

from data.fever_dataset import load_fever_splits, LABEL2ID
splits = load_fever_splits(max_train=3000, max_dev=1000, dev_test_ratio=0.1, seed=42)

# Analyze per-label evidence quality
for split_name in ["train", "dev"]:
    items = splits[split_name]
    print(f"\n{split_name} ({len(items)} examples):")
    for label in ["SUPPORTS", "REFUTES", "NOT ENOUGH INFO"]:
        label_items = [it for it in items if it["label"] == label]
        has_ev = [it for it in label_items if len(it.get("gold_evidence_text", "")) > 30]
        pct = 100 * len(has_ev) / max(1, len(label_items))
        print(f"  {label:<20}: {len(has_ev)}/{len(label_items)} ({pct:.0f}%) have evidence")
    
    # Show examples of SUPPORTS with no evidence
    no_ev_sup = [it for it in items if it["label"] == "SUPPORTS" and len(it.get("gold_evidence_text", "")) <= 30]
    if no_ev_sup:
        print(f"\n  Examples of SUPPORTS with NO evidence:")
        for it in no_ev_sup[:3]:
            print(f"    claim: {it['claim'][:80]}")
            print(f"    evidence: '{it['gold_evidence_text'][:100]}'")
            print(f"    id: {it['id']}")

# ============================================================
# DIAGNOSTIC: Check HF dataset format and evidence fields
# ============================================================
from datasets import load_dataset
ds = load_dataset("fever", "v1.0", trust_remote_code=True)
print("Train columns:", ds["train"].column_names)
print("\nFirst 3 train rows:")
for i in range(3):
    row = ds["train"][i]
    print(f"  {i}: ", {k: str(v)[:80] for k, v in row.items()})

# Check specific IDs
from data.fever_wiki_cache import WikiCache
cache = WikiCache("/content/nst/data/fever_wiki.db")

bad_ids = {169162, 191677, 124198}
found = 0
for row in ds["train"]:
    if row["id"] in bad_ids:
        found += 1
        print(f"\nID={row['id']}: {row['claim'][:60]}")
        print(f"  All keys: {list(row.keys())}")
        print(f"  Full row: {row}")
        # Check wiki cache for the evidence_wiki_url
        wiki_url = row.get("evidence_wiki_url", "")
        sent_id = row.get("evidence_sentence_id", -1)
        if wiki_url:
            in_cache = wiki_url in cache
            looked_up = cache.lookup(wiki_url)
            n_sents = len(looked_up) if looked_up else 0
            print(f"  wiki_url='{wiki_url}' sent_id={sent_id} in_cache={in_cache} n_sents={n_sents}")
            if looked_up and isinstance(sent_id, int) and 0 <= sent_id < n_sents:
                print(f"    => '{looked_up[sent_id][:120]}'")
        if found >= 3:
            break

# ============================================================
# Cell 11: PULL EVIDENCE FIX v2 (sent_id>=0 filter + sent_idx=-1 fallback)
# ============================================================
import subprocess, importlib, gc, sys

subprocess.run(["git", "pull", "--ff-only"], cwd=PROJ_ROOT, check=True)

# Force reload ALL project modules so fix takes effect
for mod in list(sys.modules.keys()):
    if any(mod.startswith(p) for p in ["data.", "models.", "training.", "eval.",
                                        "logic.", "symbolic.", "retrieval."]):
        del sys.modules[mod]
gc.collect()
torch.cuda.empty_cache()

# Reload and rebuild splits
from data.fever_dataset import load_fever_splits
splits = load_fever_splits(max_train=3000, max_dev=1000)

for split_name in ["train", "dev"]:
    items = splits[split_name]
    n_with_evidence = sum(1 for it in items if it.get("gold_evidence_text", "").strip())
    pct = n_with_evidence / len(items) * 100
    print(f"  {split_name}: {n_with_evidence}/{len(items)} ({pct:.1f}%) have evidence")

# Per-label breakdown
for split_name in ["train", "dev"]:
    print(f"\n{split_name} per-label:")
    for label in ["SUPPORTS", "REFUTES", "NOT ENOUGH INFO"]:
        items = [it for it in splits[split_name] if it["label"] == label]
        has_ev = sum(1 for it in items if it.get("gold_evidence_text", "").strip())
        print(f"  {label}: {has_ev}/{len(items)} ({has_ev/len(items)*100:.1f}%)")
print("\nEvidence fix v2 applied ✓")
# Quick diagnostic: check if the specific IDs we diagnosed now have evidence
test_ids = [169162, 191677, 124198]
for it in splits["train"]:
    if it["id"] in test_ids:
        ev = it.get("gold_evidence_text", "")
        print(f"ID {it['id']} ({it['label']}): evidence={ev[:80]!r}...")

# Also check: how many still have no evidence, broken by label?
from collections import Counter
for split_name in ["train", "dev"]:
    print(f"\n{split_name}:")
    for label in ["SUPPORTS", "REFUTES", "NOT ENOUGH INFO"]:
        items = [it for it in splits[split_name] if it["label"] == label]
        has_ev = sum(1 for it in items if it.get("gold_evidence_text", "").strip())
        print(f"  {label}: {has_ev}/{len(items)} ({has_ev/len(items)*100:.1f}%) have evidence")
# Deep debug: trace evidence resolution for ID 169162
target_id = 169162
rows = [r for r in ds["train"] if r["id"] == target_id]
print(f"Rows for {target_id}: {len(rows)}")
for r in rows[:3]:
    url = r["evidence_wiki_url"]
    sid = r["evidence_sentence_id"]
    print(f"  url={url!r} sent_id={sid}")
    sents = cache.lookup(url) if url else None
    print(f"  in_cache={sents is not None}, n_sents={len(sents) if sents else 0}")
    if sents:
        print(f"  sentence[0] = {sents[0][:100]!r}")

# Now check: does the code path actually reach our fix?
# Look at what format load_fever_splits uses
print(f"\nColumns: {ds['train'].column_names}")

# The key question: does the code use "flat" or "nested" format?
# Check if evidence_wiki_url is a string or list
r0 = ds["train"][0]
print(f"evidence_wiki_url type: {type(r0['evidence_wiki_url'])}")
print(f"evidence_sentence_id type: {type(r0['evidence_sentence_id'])}")
# ============================================================
# Cell 12: RE-RUN NEURAL BASELINE 3K (with fixed evidence)
# ============================================================
import time, json, gc, sys

for mod in list(sys.modules.keys()):
    if any(mod.startswith(p) for p in ["data.", "models.", "training.", "eval.",
                                        "logic.", "symbolic.", "retrieval."]):
        del sys.modules[mod]
gc.collect()
torch.cuda.empty_cache()

print("=" * 65)
print("  3K NEURAL BASELINE v2: DeBERTa-v3-large + LoRA (fixed evidence)")
print("=" * 65)

t0 = time.time()
from training.train_fever_nst import train_fever_nst
results_neural_3k_v2 = train_fever_nst(
    "configs/fever_neural_3k_a100.yaml",
    config_overrides=GPU_OVERRIDES,
)
elapsed = time.time() - t0

dev = results_neural_3k_v2.get("dev", {})
print(f"\n{'='*65}")
print(f"  NEURAL BASELINE 3K v2 RESULTS ({elapsed/60:.1f} min)")
print(f"{'='*65}")
print(f"  Dev acc    : {dev.get('accuracy', 'N/A')}")
print(f"  Best dev   : {results_neural_3k_v2.get('best_dev_acc', 'N/A')}")
for label, stats in dev.get("per_label", {}).items():
    print(f"    {label:<20}: {stats.get('accuracy', 0):.4f} (n={stats.get('count', 0)})")

with open("results_neural_3k_v2.json", "w") as f:
    json.dump(results_neural_3k_v2, f, indent=2, default=str)
print(f"\n  Saved to results_neural_3k_v2.json")
# ============================================================
# Cell 13: RE-RUN NST-VERI 3K (with fixed evidence)
# ============================================================
import time, json, gc, sys

for mod in list(sys.modules.keys()):
    if any(mod.startswith(p) for p in ["data.", "models.", "training.", "eval.",
                                        "logic.", "symbolic.", "retrieval."]):
        del sys.modules[mod]
gc.collect()
torch.cuda.empty_cache()

print("=" * 65)
print("  3K NST-VERI v2: DeBERTa-v3-large + LoRA + Constraints (fixed evidence)")
print("=" * 65)

t0 = time.time()
from training.train_fever_veri import train_fever_veri
results_veri_3k_v2 = train_fever_veri(
    "configs/fever_veri_3k_a100.yaml",
    config_overrides=GPU_OVERRIDES,
)
elapsed = time.time() - t0

dev = results_veri_3k_v2.get("dev", {})
print(f"\n{'='*65}")
print(f"  NST-VERI 3K v2 RESULTS ({elapsed/60:.1f} min)")
print(f"{'='*65}")
print(f"  Dev acc    : {dev.get('accuracy', 'N/A')}")
print(f"  Best dev   : {results_veri_3k_v2.get('best_dev_acc', 'N/A')}")
for label, stats in dev.get("per_label", {}).items():
    print(f"    {label:<20}: {stats.get('accuracy', 0):.4f} (n={stats.get('count', 0)})")

# Compare with neural
n_best = results_neural_3k_v2.get("best_dev_acc", 0)
v_best = results_veri_3k_v2.get("best_dev_acc", 0)
delta = v_best - n_best
print(f"\nDelta (VERI - Neural): {delta:+.4f}")
if v_best >= 0.90:
    print("*** TARGET REACHED: 90%+ accuracy! ***")

with open("results_veri_3k_v2.json", "w") as f:
    json.dump(results_veri_3k_v2, f, indent=2, default=str)
print(f"  Saved to results_veri_3k_v2.json")
# Quick results summary
import json
with open("results_veri_3k_v2.json") as f:
    r = json.load(f)
dev = r.get("dev", {})
print(f"NST-VERI 3K v2:")
print(f"  Best dev acc: {r.get('best_dev_acc', 'N/A')}")
print(f"  Final dev acc: {dev.get('accuracy', 'N/A')}")
for label, stats in dev.get("per_label", {}).items():
    print(f"    {label}: {stats.get('accuracy', 0):.4f} (n={stats.get('count', 0)})")

n_best = results_neural_3k_v2.get("best_dev_acc", 0)
v_best = r.get("best_dev_acc", 0)
print(f"\nNeural best: {n_best:.4f}")
print(f"VERI best:   {v_best:.4f}")
print(f"Delta:       {v_best - n_best:+.4f}")
if v_best >= 0.90:
    print("*** TARGET REACHED: 90%+ ***")
# ============================================================
# PHASE 0: ENVIRONMENT GATE — Must be remote A100
# ============================================================
import torch, os, subprocess

assert torch.cuda.is_available(), "BLOCKED: No CUDA. Must run on A100."
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {gpu_name} ({vram_gb:.0f} GB)")
assert "A100" in gpu_name or vram_gb >= 35, f"Expected A100, got {gpu_name}"

# PHASE 1: Pull latest code
subprocess.run(["git", "pull", "--ff-only"], cwd=PROJ_ROOT, check=True)
print(f"Git HEAD: ", end="")
subprocess.run(["git", "log", "--oneline", "-1"], cwd=PROJ_ROOT)

# Reload all project modules
import gc, sys
for mod in list(sys.modules.keys()):
    if any(mod.startswith(p) for p in ["data.", "models.", "training.", "eval.",
                                        "logic.", "symbolic.", "retrieval."]):
        del sys.modules[mod]
gc.collect()
torch.cuda.empty_cache()

print(f"\nEnvironment verified. Ready for 10K experiments.")
# ============================================================
# 10K NEURAL BASELINE: DeBERTa-v3-large + LoRA (NO constraints)
# ============================================================
# 12k train / 3k dev / dev_test_ratio=0.1 (300 held-out)
# A100-optimized: bs=32, grad_accum=2, bf16, 4 workers
import time, json, gc, sys, torch

for mod in list(sys.modules.keys()):
    if any(mod.startswith(p) for p in ["data.", "models.", "training.", "eval.",
                                        "logic.", "symbolic.", "retrieval."]):
        del sys.modules[mod]
gc.collect()
torch.cuda.empty_cache()

print("=" * 65)
print("  10K NEURAL BASELINE: DeBERTa-v3-large + LoRA (NO constraints)")
print("  Train: 12k | Dev: 3k | Held-out: 300 | Epochs: 5")
print("  A100 optimized: bs=32×2=64 effective, BF16, TF32")
print("=" * 65)

t0 = time.time()
from training.train_fever_nst import train_fever_nst
results_neural_10k = train_fever_nst("configs/fever_neural_10k_a100.yaml")
elapsed = time.time() - t0

dev = results_neural_10k.get("dev", {})
print(f"\n{'='*65}")
print(f"  NEURAL BASELINE 10K RESULTS ({elapsed/60:.1f} min)")
print(f"{'='*65}")
print(f"  Dev acc    : {dev.get('accuracy', 'N/A')}")
print(f"  Dev ECE    : {dev.get('ece', 'N/A')}")
print(f"  Best dev   : {results_neural_10k.get('best_dev_acc', 'N/A')}")
for label, stats in dev.get("per_label", {}).items():
    print(f"    {label:<20}: {stats.get('accuracy', 0):.4f} (n={stats.get('count', 0)})")

# Held-out dev_test
dt = results_neural_10k.get("dev_test")
if dt:
    print(f"\n  Held-out dev_test accuracy: {dt.get('accuracy', '?')}")

with open("results_neural_10k.json", "w") as f:
    json.dump(results_neural_10k, f, indent=2, default=str)
print(f"\n  Saved to results_neural_10k.json")
# ============================================================
# 10K NST-VERI: DeBERTa-v3-large + LoRA + 7 Constraints
# ============================================================
# Same 12k/3k split. 3-phase training with uncertainty-focused constraints.
# v2.1 constraint engine: 7 high-precision constraints
import time, json, gc, sys, torch

for mod in list(sys.modules.keys()):
    if any(mod.startswith(p) for p in ["data.", "models.", "training.", "eval.",
                                        "logic.", "symbolic.", "retrieval."]):
        del sys.modules[mod]
gc.collect()
torch.cuda.empty_cache()

print("=" * 65)
print("  10K NST-VERI: DeBERTa-v3-large + LoRA + 7 Constraints")
print("  Train: 12k | Dev: 3k | Held-out: 300 | Epochs: 8")
print("  3-phase: NLI+aux → +contrastive → +uncertainty-focused constraints")
print("  A100 optimized: bs=32×2=64 effective, BF16, TF32, focal loss")
print("=" * 65)

t0 = time.time()
from training.train_fever_veri import train_fever_veri
results_veri_10k = train_fever_veri("configs/fever_veri_10k_a100.yaml")
elapsed = time.time() - t0

dev = results_veri_10k.get("dev", {})
print(f"\n{'='*65}")
print(f"  NST-VERI 10K RESULTS ({elapsed/60:.1f} min)")
print(f"{'='*65}")
print(f"  Dev acc    : {dev.get('accuracy', 'N/A')}")
print(f"  Dev ECE    : {dev.get('ece', 'N/A')}")
print(f"  Best dev   : {results_veri_10k.get('best_dev_acc', 'N/A')}")
for label, stats in dev.get("per_label", {}).items():
    print(f"    {label:<20}: {stats.get('accuracy', 0):.4f} (n={stats.get('count', 0)})")

# Constraint diagnostics
train_log = results_veri_10k.get("train_log", [])
phase3 = [e for e in train_log if e.get("phase", 0) >= 3]
if phase3:
    cst = [e.get("loss_constraint", 0) for e in phase3]
    lam = [e.get("mean_lambda", 0) for e in phase3]
    fire = [e.get("fire_rate", 0) for e in phase3]
    print(f"\n  CONSTRAINT DIAGNOSTICS (Phase 3):")
    print(f"    Entries           : {len(phase3)}")
    print(f"    Constraint loss   : {min(cst):.4f}—{max(cst):.4f} (mean {sum(cst)/len(cst):.4f})")
    print(f"    Mean lambda       : {min(lam):.4f}—{max(lam):.4f}")
    print(f"    Fire rate         : {min(fire):.4f}—{max(fire):.4f}")
    if max(cst) > 0.001:
        print(f"    ✓ CONSTRAINTS ARE ACTIVE")
    else:
        print(f"    ✗ WARNING: CONSTRAINTS STILL INACTIVE")

# Held-out dev_test
dt = results_veri_10k.get("dev_test")
if dt:
    print(f"\n  Held-out dev_test accuracy: {dt.get('accuracy', '?')}")

with open("results_veri_10k.json", "w") as f:
    json.dump(results_veri_10k, f, indent=2, default=str)
print(f"\n  Saved to results_veri_10k.json")
# ============================================================
# 10K COMPARISON — Neural vs NST-VERI (HONEST REPORT)
# ============================================================
import json, os

experiments_10k = {}
for name, path in [("neural_10k", "results_neural_10k.json"),
                   ("veri_10k", "results_veri_10k.json")]:
    if os.path.exists(path):
        with open(path) as f:
            experiments_10k[name] = json.load(f)

print(f"{'='*70}")
print(f"  10K COMPARISON: Neural vs NST-VERI (FEVER Gold Evidence)")
print(f"{'='*70}")
print(f"  {'Method':<25} {'Dev Acc':>10} {'Best Acc':>10} {'ECE':>8} {'DevTest':>10}")
print(f"  {'─'*63}")

for name, r in experiments_10k.items():
    dev = r.get("dev", {})
    acc = dev.get("accuracy", "?")
    best = r.get("best_dev_acc", "?")
    ece = dev.get("ece", "?")
    dt = r.get("dev_test", {})
    dt_acc = dt.get("accuracy", "?") if dt else "?"
    fmt = lambda v: f"{v:.4f}" if isinstance(v, (int, float)) else str(v)
    print(f"  {name:<25} {fmt(acc):>10} {fmt(best):>10} {fmt(ece):>8} {fmt(dt_acc):>10}")

# Per-label breakdown
for name, r in experiments_10k.items():
    dev = r.get("dev", {})
    print(f"\n  {name} per-label:")
    for label, stats in dev.get("per_label", {}).items():
        print(f"    {label:<20}: {stats.get('accuracy', 0):.4f} (n={stats.get('count', 0)})")

# Delta
if "neural_10k" in experiments_10k and "veri_10k" in experiments_10k:
    n_best = experiments_10k["neural_10k"].get("best_dev_acc", 0)
    v_best = experiments_10k["veri_10k"].get("best_dev_acc", 0)
    delta = v_best - n_best
    print(f"\n{'='*70}")
    print(f"  Delta (VERI - Neural): {delta:+.4f}")
    if delta > 0.02:
        print(f"  RESULT: NST-VERI shows meaningful improvement over neural baseline")
    elif delta > 0:
        print(f"  RESULT: NST-VERI shows marginal improvement (within noise)")
    elif delta > -0.02:
        print(f"  RESULT: Neural and VERI are approximately tied")
    else:
        print(f"  RESULT: Neural baseline outperforms VERI")
    
    # Verify no leakage
    print(f"\n  INTEGRITY:")
    print(f"    Train/dev split: separate (dev_test_ratio=0.1 held out)")
    print(f"    Same backbone: DeBERTa-v3-large + LoRA for both")
    print(f"    Same data: 12k train / 3k dev for both")
    print(f"    Same seed: 42")
    print(f"    Evaluation: on dev set only, NOT on train set")
print(f"{'='*70}")
# ============================================================
# Cell 10: FULL NEURAL BASELINE (Fair Comparison)
# ============================================================
# Same DeBERTa-v3-large + LoRA, NO constraints.
import time, json, gc, sys

for mod in list(sys.modules.keys()):
    if any(mod.startswith(p) for p in ["data.", "models.", "training.", "eval.",
                                        "logic.", "symbolic.", "retrieval."]):
        del sys.modules[mod]
gc.collect()
torch.cuda.empty_cache()

print("=" * 65)
print("  FULL NEURAL BASELINE: DeBERTa-v3-large + LoRA")
print("=" * 65)

t0 = time.time()
from training.train_fever_nst import train_fever_nst
results_neural_full = train_fever_nst(
    "configs/fever_gold_neural.yaml",
    config_overrides=GPU_OVERRIDES,
)
elapsed = time.time() - t0

dev = results_neural_full.get("dev", {})
print(f"\n{'='*65}")
print(f"  FULL NEURAL RESULTS ({elapsed/60:.1f} min)")
print(f"{'='*65}")
print(f"  Dev acc    : {dev.get('accuracy', 'N/A')}")
print(f"  Best dev   : {results_neural_full.get('best_dev_acc', 'N/A')}")
for label, stats in dev.get("per_label", {}).items():
    print(f"    {label:<20}: {stats.get('accuracy', 0):.4f} (n={stats.get('count', 0)})")

with open("results_neural_full.json", "w") as f:
    json.dump(results_neural_full, f, indent=2, default=str)
print(f"\n  Saved to results_neural_full.json")
# ============================================================
# Cell 11: FINAL COMPARISON & HONEST REPORT
# ============================================================
import json, os, glob

experiments = {}
for name, path in [
    ("neural_3k", "results_neural_3k.json"),
    ("veri_3k", "results_veri_3k.json"),
    ("neural_full", "results_neural_full.json"),
    ("veri_full", "results_veri_full.json"),
]:
    if os.path.exists(path):
        with open(path) as f:
            experiments[name] = json.load(f)

print(f"{'='*65}")
print(f"  FINAL RESULTS — FEVER Gold Evidence Label Accuracy")
print(f"{'='*65}")
print(f"  {'Method':<25} {'Dev Acc':>10} {'Best Acc':>10} {'ECE':>8} {'Brier':>8}")
print(f"  {'─'*63}")

for name, r in sorted(experiments.items()):
    dev = r.get("dev", {})
    acc = dev.get("accuracy", "?")
    best = r.get("best_dev_acc", "?")
    ece = dev.get("ece", "?")
    brier = dev.get("brier", "?")
    fmt = lambda v: f"{v:.4f}" if isinstance(v, (int, float)) else str(v)
    print(f"  {name:<25} {fmt(acc):>10} {fmt(best):>10} {fmt(ece):>8} {fmt(brier):>8}")

# Per-label breakdown for full runs
for name in ["neural_full", "veri_full"]:
    if name in experiments:
        dev = experiments[name].get("dev", {})
        print(f"\n  {name} per-label:")
        for label, stats in dev.get("per_label", {}).items():
            print(f"    {label:<20}: {stats.get('accuracy', 0):.4f} (n={stats.get('count', 0)})")

# Held-out dev_test comparison
for name in ["neural_full", "veri_full"]:
    if name in experiments:
        dt = experiments[name].get("dev_test")
        if dt:
            print(f"\n  {name} held-out dev_test: {dt.get('accuracy', '?')}")

print(f"\n{'='*65}")
print(f"  HONEST ASSESSMENT")
print(f"{'='*65}")
if "veri_full" in experiments:
    final_acc = experiments["veri_full"].get("best_dev_acc", 0)
    if final_acc >= 0.90:
        print(f"  TARGET REACHED: {final_acc:.4f} >= 0.90")
    else:
        print(f"  TARGET NOT YET REACHED: {final_acc:.4f} < 0.90")
        print(f"  Next steps: analyze per-label failures, tune constraints, retrain")
# ============================================================
# Cell 12: SEED SWEEP (Reproducibility — 3 seeds)
# ============================================================
# Run after achieving 90%+ on seed=42 to verify result is real.
import time, json, gc, sys

seeds = [42, 43, 44]
seed_results = {}

for seed in seeds:
    for mod in list(sys.modules.keys()):
        if any(mod.startswith(p) for p in ["data.", "models.", "training.", "eval.",
                                            "logic.", "symbolic.", "retrieval."]):
            del sys.modules[mod]
    gc.collect()
    torch.cuda.empty_cache()

    print(f"\n{'='*50}")
    print(f"  Seed {seed} — Full NST-VERI")
    print(f"{'='*50}")

    t0 = time.time()
    from training.train_fever_veri import train_fever_veri
    r = train_fever_veri(
        "configs/fever_gold_nst_veri_a100.yaml",
        config_overrides={
            **GPU_OVERRIDES,
            "seed": seed,
            "io": {"out_dir": f"outputs_veri_seed{seed}"},
        },
    )
    elapsed = time.time() - t0
    seed_results[seed] = r
    dev = r.get("dev", {})
    print(f"  Seed {seed}: acc={dev.get('accuracy','?')} best={r.get('best_dev_acc','?')} ({elapsed/60:.1f}min)")

# Summary
accs = [seed_results[s].get("best_dev_acc", 0) for s in seeds]
import numpy as np
print(f"\n{'='*50}")
print(f"  SEED SWEEP RESULTS")
print(f"  Accs: {accs}")
print(f"  Mean: {np.mean(accs):.4f} ± {np.std(accs):.4f}")
print(f"{'='*50}")
# ============================================================
# Cell 13: SAVE ARTIFACTS — Download from Colab
# ============================================================
import json, os, shutil, glob

# Gather all result files
artifacts = glob.glob("results_*.json") + glob.glob("outputs_*/report.json")
print(f"Result artifacts: {artifacts}")

# Create a zip for easy download
artifact_dir = "nst_results"
os.makedirs(artifact_dir, exist_ok=True)
for f in artifacts:
    shutil.copy(f, artifact_dir)
# Add configs used
for cfg in glob.glob("configs/fever_*3k*.yaml") + glob.glob("configs/fever_gold_nst_veri*.yaml"):
    shutil.copy(cfg, artifact_dir)

shutil.make_archive("nst_results", "zip", ".", artifact_dir)
print(f"Download: nst_results.zip")

# Colab download helper
try:
    from google.colab import files
    files.download("nst_results.zip")
except ImportError:
    print("Not on Colab — download nst_results.zip manually")
# ============================================================
# Cell 14: LEAKAGE AUDIT — Verify no data contamination
# ============================================================
# Run AFTER training to verify results are honest.
import json, sys, os

for mod in list(sys.modules.keys()):
    if any(mod.startswith(p) for p in ["data.", "models.", "training.", "eval.",
                                        "logic.", "symbolic.", "retrieval."]):
        del sys.modules[mod]

from data.fever_dataset import load_fever_splits

# Load same splits with same seed
splits = load_fever_splits(max_train=None, max_dev=None, dev_test_ratio=0.1, seed=42)

train_claims = {it["claim"] for it in splits["train"]}
dev_claims = {it["claim"] for it in splits["dev"]}
dev_test_claims = {it["claim"] for it in splits.get("dev_test", [])}

# Check overlaps
train_dev_overlap = train_claims & dev_claims
train_devtest_overlap = train_claims & dev_test_claims
dev_devtest_overlap = dev_claims & dev_test_claims

print(f"{'='*50}")
print(f"  LEAKAGE AUDIT")
print(f"{'='*50}")
print(f"  Train size     : {len(splits['train'])}")
print(f"  Dev size       : {len(splits['dev'])}")
print(f"  Dev-test size  : {len(splits.get('dev_test', []))}")
print(f"  Train∩Dev      : {len(train_dev_overlap)} overlapping claims")
print(f"  Train∩DevTest  : {len(train_devtest_overlap)} overlapping claims")
print(f"  Dev∩DevTest    : {len(dev_devtest_overlap)} overlapping claims")

if len(train_dev_overlap) == 0 and len(train_devtest_overlap) == 0:
    print(f"\n  LEAKAGE CHECK PASSED — No contamination detected")
else:
    print(f"\n  WARNING: Potential leakage detected!")
    if train_dev_overlap:
        print(f"    Example overlap: {list(train_dev_overlap)[:3]}")

In [115]:
# ============================================================
# Cell 3: 10K NEURAL BASELINE (subprocess — avoids import conflicts)
# ============================================================
import subprocess, sys, json, os, shutil, time

# Clean previous run
outdir = "/content/nst/outputs_neural_10k"
if os.path.isdir(outdir):
    shutil.rmtree(outdir)
    print(f"Cleaned previous: {outdir}")

print("=" * 65)
print("  10K NEURAL BASELINE: DeBERTa-v3-large + LoRA (NO constraints)")
print("  Train: 12k | Dev: 3k | Held-out: 300 | Epochs: 5")
print("  A100: bs=32×2=64 effective, FP32+TF32")
print("=" * 65)

t0 = time.time()
ret = subprocess.run(
    [sys.executable, "main.py", "train-fever-nst",
     "--config", "configs/fever_neural_10k_a100.yaml",
     "--outdir", "outputs_neural_10k"],
    cwd="/content/nst"
)
elapsed = time.time() - t0
print(f"\nTraining took {elapsed/60:.1f} min")

if ret.returncode != 0:
    raise RuntimeError(f"Neural baseline failed with code {ret.returncode}")

# Load saved report
with open("/content/nst/outputs_neural_10k/report.json") as f:
    results_neural = json.load(f)

with open("/content/nst/results_neural_10k.json", "w") as f:
    json.dump(results_neural, f, indent=2)

print(f"\n{'='*65}")
print(f"  NEURAL BASELINE 10K RESULTS ({elapsed/60:.1f} min)")
print(f"{'='*65}")
dev = results_neural.get("dev", {})
print(f"  Dev acc    : {dev.get('accuracy', '?')}")
print(f"  Dev ECE    : {dev.get('ece', '?')}")
print(f"  Best dev   : {results_neural.get('best_dev_acc', '?')}")
for label, stats in dev.get("per_label", {}).items():
    print(f"    {label:<20}: {stats['accuracy']:.4f} (n={stats['count']})")
dt = results_neural.get("dev_test", {})
if dt:
    print(f"\n  Held-out dev_test accuracy: {dt.get('accuracy', '?')}")
print(f"\n  Saved to results_neural_10k.json")

Cleaned previous: /content/nst/outputs_neural_10k
  10K NEURAL BASELINE: DeBERTa-v3-large + LoRA (NO constraints)
  Train: 12k | Dev: 3k | Held-out: 300 | Epochs: 5
  A100: bs=32×2=64 effective, FP32+TF32

Training took 16.1 min

  NEURAL BASELINE 10K RESULTS (16.1 min)
  Dev acc    : 0.8704
  Dev ECE    : 0.04892
  Best dev   : 0.8704
    SUPPORTS            : 0.9042 (n=898)
    REFUTES             : 0.8115 (n=870)
    NOT ENOUGH INFO     : 0.8927 (n=932)

  Held-out dev_test accuracy: 0.84

  Saved to results_neural_10k.json


In [119]:
# ============================================================
# Cell 4: 10K NST-VERI v0.4 (subprocess — avoids import conflicts)
# ============================================================
import subprocess, sys, json, os

print("=" * 65)
print("  10K NST-VERI v0.4: evidence-gated + inference-time fusion")
print("  Train: 12k | Dev: 3k | Held-out: 300 | Epochs: 8")
print("  Changes: evidence gating, simplified constraint loss,")
print("           inference-time constraint fusion, λ_max=0.5, β_aux=0.3")
print("  A100 optimized: bs=32×2=64 effective, BF16, TF32, focal loss")
print("=" * 65)

ret = subprocess.run(
    [sys.executable, "main.py", "train-fever-veri",
     "--config", "configs/fever_veri_10k_a100.yaml",
     "--outdir", "outputs_veri_10k"],
    cwd="/content/nst"
)
if ret.returncode != 0:
    raise RuntimeError(f"NST-VERI failed with code {ret.returncode}")

# Load saved report
with open("/content/nst/outputs_veri_10k/report.json") as f:
    results_veri = json.load(f)

# Also save a copy at project root
with open("/content/nst/results_veri_10k.json", "w") as f:
    json.dump(results_veri, f, indent=2)

print(f"\n{'='*65}")
print(f"  NST-VERI 10K RESULTS ({results_veri.get('elapsed_s', 0)/60:.1f} min)")
print(f"{'='*65}")
dev = results_veri.get("dev", {})
print(f"  Dev acc (fused): {dev.get('accuracy', '?')}")
print(f"  Dev ECE        : {dev.get('ece', '?')}")
print(f"  Best dev (raw) : {results_veri.get('best_dev_acc', '?')}")
for label, stats in dev.get("per_label", {}).items():
    print(f"    {label:<20}: {stats['accuracy']:.4f} (n={stats['count']})")
dt = results_veri.get("dev_test", {})
if dt:
    print(f"\n  Held-out dev_test accuracy: {dt.get('accuracy', '?')}")
print(f"\n  Saved to results_veri_10k.json")

  10K NST-VERI v0.4: evidence-gated + inference-time fusion
  Train: 12k | Dev: 3k | Held-out: 300 | Epochs: 8
  Changes: evidence gating, simplified constraint loss,
           inference-time constraint fusion, λ_max=0.5, β_aux=0.3
  A100 optimized: bs=32×2=64 effective, BF16, TF32, focal loss

  NST-VERI 10K RESULTS (21.8 min)
  Dev acc (fused): 0.8737
  Dev ECE        : 0.057649
  Best dev (raw) : 0.8737
    SUPPORTS            : 0.9232 (n=898)
    REFUTES             : 0.7943 (n=870)
    NOT ENOUGH INFO     : 0.9002 (n=932)

  Held-out dev_test accuracy: 0.8433

  Saved to results_veri_10k.json


In [120]:
# ============================================================
# Cell 5: 10K COMPARISON — Neural vs NST-VERI v0.4 (HONEST REPORT)
# ============================================================
import json, os

experiments = {}
for name, path in [("neural_10k", "results_neural_10k.json"),
                   ("veri_10k", "results_veri_10k.json")]:
    if os.path.exists(path):
        with open(path) as f:
            experiments[name] = json.load(f)

print(f"{'='*70}")
print(f"  10K COMPARISON: Neural vs NST-VERI v0.4 (FEVER Gold Evidence)")
print(f"{'='*70}")
print(f"  {'Method':<25} {'Dev Acc':>10} {'Best Acc':>10} {'ECE':>8} {'DevTest':>10}")
print(f"  {'─'*63}")

for name, r in experiments.items():
    dev = r.get("dev", {})
    acc = dev.get("accuracy", "?")
    best = r.get("best_dev_acc", "?")
    ece = dev.get("ece", "?")
    dt = r.get("dev_test", {})
    dt_acc = dt.get("accuracy", "?") if dt else "?"
    fmt = lambda v: f"{v:.4f}" if isinstance(v, (int, float)) else str(v)
    print(f"  {name:<25} {fmt(acc):>10} {fmt(best):>10} {fmt(ece):>8} {fmt(dt_acc):>10}")

# Per-label breakdown
for name, r in experiments.items():
    dev = r.get("dev", {})
    print(f"\n  {name} per-label:")
    for label, stats in dev.get("per_label", {}).items():
        print(f"    {label:<20}: {stats.get('accuracy', 0):.4f} (n={stats.get('count', 0)})")

# Delta
if "neural_10k" in experiments and "veri_10k" in experiments:
    n_dev = experiments["neural_10k"].get("dev", {}).get("accuracy", 0)
    v_dev = experiments["veri_10k"].get("dev", {}).get("accuracy", 0)
    n_held = experiments["neural_10k"].get("dev_test", {}).get("accuracy", 0) if experiments["neural_10k"].get("dev_test") else 0
    v_held = experiments["veri_10k"].get("dev_test", {}).get("accuracy", 0) if experiments["veri_10k"].get("dev_test") else 0
    n_best = experiments["neural_10k"].get("best_dev_acc", 0)
    v_best = experiments["veri_10k"].get("best_dev_acc", 0)
    
    print(f"\n{'='*70}")
    print(f"  DELTAS (VERI - Neural):")
    print(f"    Dev accuracy:     {v_dev - n_dev:+.4f}")
    print(f"    Best dev:         {v_best - n_best:+.4f}")
    print(f"    Held-out:         {v_held - n_held:+.4f}")
    
    if v_dev > n_dev and v_held > n_held:
        print(f"\n  ✅ NST-VERI v0.4 SUPERIOR on both dev and held-out")
    elif v_dev > n_dev or v_held > n_held:
        print(f"\n  ⚠️ NST-VERI v0.4 wins on one metric, mixed on the other")
    else:
        print(f"\n  ❌ Neural baseline still ahead — need further improvements")
    
    print(f"\n  INTEGRITY:")
    print(f"    Train/dev split: separate (dev_test_ratio=0.1 held out)")
    print(f"    Same backbone: DeBERTa-v3-large + LoRA for both")
    print(f"    Same data: 12k train / 3k dev for both")
    print(f"    Same seed: 42")
    print(f"    Evaluation: dev set only, NOT on train set")
    print(f"    VERI uses inference-time constraint fusion (honest advantage)")
print(f"{'='*70}")

  10K COMPARISON: Neural vs NST-VERI v0.4 (FEVER Gold Evidence)
  Method                       Dev Acc   Best Acc      ECE    DevTest
  ───────────────────────────────────────────────────────────────
  neural_10k                    0.8704     0.8704   0.0489     0.8400
  veri_10k                      0.8737     0.8737   0.0576     0.8433

  neural_10k per-label:
    SUPPORTS            : 0.9042 (n=898)
    REFUTES             : 0.8115 (n=870)
    NOT ENOUGH INFO     : 0.8927 (n=932)

  veri_10k per-label:
    SUPPORTS            : 0.9232 (n=898)
    REFUTES             : 0.7943 (n=870)
    NOT ENOUGH INFO     : 0.9002 (n=932)

  DELTAS (VERI - Neural):
    Dev accuracy:     +0.0033
    Best dev:         +0.0033
    Held-out:         +0.0033

  ✅ NST-VERI v0.4 SUPERIOR on both dev and held-out

  INTEGRITY:
    Train/dev split: separate (dev_test_ratio=0.1 held out)
    Same backbone: DeBERTa-v3-large + LoRA for both
    Same data: 12k train / 3k dev for both
    Same seed: 42
    Evalu

# NST-VERI v2 Redesign — Training & Evaluation

**V2 Architecture Changes:**
- FocalCrossEntropy with per-class gamma (REFUTES γ=3.0)
- Separate AttentionPool per aux head (complementary features)
- Signal-only RecalibrationNetwork (7-dim, not 768-dim CLS)
- Symmetric R-Drop (all losses averaged)
- Supervised contrastive with class prototypes
- Two-phase warmup schedule

**Run order:**
1. Environment gate
2. Git pull latest code
3. Neural baseline 10K (if not already run)
4. NST-VERI v2 10K
5. Comparison

In [1]:
# ============================================================
# Cell 8: ENVIRONMENT GATE + GIT PULL (v2)
# ============================================================
import os, sys, torch

_cwd = os.getcwd()
if not _cwd.startswith("/content"):
    os.chdir("/content/nst")
    _cwd = os.getcwd()

assert torch.cuda.is_available(), "CUDA required — use Colab A100"
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"  GPU: {gpu_name} ({vram_gb:.1f} GB)")
print(f"  PyTorch: {torch.__version__}")
print(f"  CUDA: {torch.version.cuda}")

import subprocess
r = subprocess.run(["git", "pull", "--ff-only"], capture_output=True, text=True, cwd="/content/nst")
print(f"  Git pull: {r.stdout.strip()}")
if r.returncode != 0:
    print(f"  Git error: {r.stderr}")

# Verify v2 files exist
for f in ["models/nst_veri_v2.py", "training/train_fever_veri_v2.py",
          "configs/fever_veri_v2_10k_a100.yaml"]:
    assert os.path.exists(f), f"Missing: {f}"
print("  All v2 files present ✓")

  GPU: NVIDIA A100-SXM4-40GB (42.4 GB)
  PyTorch: 2.9.0+cu126
  CUDA: 12.6


FileNotFoundError: [Errno 2] No such file or directory: '/content/nst'

In [ ]:
# ============================================================
# Cell 9: 10K NEURAL BASELINE (subprocess)
# ============================================================
# Run this if you don't already have a neural 10K result.
import subprocess, sys, json, os, shutil, time

outdir = "/content/nst/outputs_neural_10k"
if os.path.exists(outdir):
    shutil.rmtree(outdir)

print("=" * 65)
print("  10K NEURAL BASELINE — DeBERTa-v3-base + LoRA")
print("=" * 65)
t0 = time.time()

result = subprocess.run(
    [sys.executable, "main.py", "train-fever-nst",
     "--config", "configs/fever_neural_10k_a100.yaml",
     "--outdir", outdir],
    cwd="/content/nst",
    capture_output=False, text=True,
    timeout=3600,
)

elapsed = time.time() - t0
print(f"\n  Neural baseline done in {elapsed:.0f}s (exit={result.returncode})")

# Load and display results
report_path = os.path.join(outdir, "report.json")
if os.path.exists(report_path):
    report = json.load(open(report_path))
    print(f"\n  Dev accuracy:  {report.get('dev', {}).get('accuracy', 'N/A')}")
    if 'dev_test' in report and report['dev_test']:
        print(f"  Held-out acc:  {report['dev_test'].get('accuracy', 'N/A')}")
    per_label = report.get('dev', {}).get('per_label', {})
    for lbl, stats in per_label.items():
        print(f"    {lbl}: {stats.get('accuracy', 'N/A')}")
    # Save for comparison
    json.dump(report, open("/content/nst/results_neural_10k.json", "w"), indent=2)
else:
    print("  WARNING: No report.json found")

In [ ]:
# ============================================================
# Cell 10: 10K NST-VERI v2 (subprocess)
# ============================================================
import subprocess, sys, json, os, shutil, time

outdir = "/content/nst/outputs_veri_v2_10k"
if os.path.exists(outdir):
    shutil.rmtree(outdir)

print("=" * 65)
print("  10K NST-VERI v2 — Learned Multi-Task + Focal Loss")
print("  FocalCE (REFUTES γ=3.0), Signal-only Recalib, Symmetric R-Drop")
print("=" * 65)
t0 = time.time()

result = subprocess.run(
    [sys.executable, "main.py", "train-fever-veri-v2",
     "--config", "configs/fever_veri_v2_10k_a100.yaml",
     "--outdir", outdir],
    cwd="/content/nst",
    capture_output=False, text=True,
    timeout=7200,
)

elapsed = time.time() - t0
print(f"\n  NST-VERI v2 done in {elapsed:.0f}s (exit={result.returncode})")

# Load and display results
report_path = os.path.join(outdir, "report.json")
if os.path.exists(report_path):
    report = json.load(open(report_path))
    print(f"\n  Best dev accuracy:  {report.get('best_dev_acc', 'N/A')}")
    print(f"  Temperature:        {report.get('temperature', 'N/A')}")
    print(f"  NaN abort:          {report.get('nan_abort', False)}")
    print(f"  Early stop:         {report.get('early_stop', False)}")

    dev = report.get('dev', {})
    print(f"\n  Final dev accuracy (+ symbolic): {dev.get('accuracy', 'N/A')}")
    print(f"  ECE: {dev.get('ece', 'N/A')}")
    for lbl, stats in dev.get('per_label', {}).items():
        print(f"    {lbl}: {stats.get('accuracy', 'N/A')} (n={stats.get('count', '?')})")

    dev_raw = report.get('dev_raw', {})
    print(f"\n  Raw dev accuracy (no symbolic): {dev_raw.get('accuracy', 'N/A')}")

    dev_test = report.get('dev_test')
    if dev_test:
        print(f"\n  Held-out accuracy: {dev_test.get('accuracy', 'N/A')}")
        for lbl, stats in dev_test.get('per_label', {}).items():
            print(f"    {lbl}: {stats.get('accuracy', 'N/A')} (n={stats.get('count', '?')})")

    # Save for comparison
    json.dump(report, open("/content/nst/results_veri_v2_10k.json", "w"), indent=2)
else:
    print("  WARNING: No report.json found")

In [ ]:
# ============================================================
# Cell 11: 10K COMPARISON — Neural vs NST-VERI v2 (HONEST REPORT)
# ============================================================
import json, os

experiments = {}
for name, path in [("neural_10k", "results_neural_10k.json"),
                   ("veri_v2_10k", "results_veri_v2_10k.json")]:
    fpath = os.path.join("/content/nst", path)
    if os.path.exists(fpath):
        experiments[name] = json.load(open(fpath))
    else:
        print(f"  Missing: {path}")

if len(experiments) < 2:
    print("  Need both neural and v2 results for comparison")
else:
    print("=" * 70)
    print("  10K COMPARISON: Neural Baseline vs NST-VERI v2")
    print("=" * 70)

    # Helper
    def get_acc(report, split="dev"):
        s = report.get(split, {})
        return s.get("accuracy", 0) if isinstance(s, dict) else 0

    def get_per_label(report, split="dev"):
        s = report.get(split, {})
        if isinstance(s, dict):
            return s.get("per_label", {})
        return {}

    neural = experiments["neural_10k"]
    veri = experiments["veri_v2_10k"]

    n_dev = get_acc(neural, "dev")
    v_dev = get_acc(veri, "dev")
    delta_dev = v_dev - n_dev

    print(f"\n  {'Metric':<25} {'Neural':>10} {'VERI v2':>10} {'Delta':>10}")
    print(f"  {'-'*55}")
    print(f"  {'Dev Accuracy':<25} {n_dev:>10.4f} {v_dev:>10.4f} {delta_dev:>+10.4f}")

    # Held-out
    n_ht = get_acc(neural, "dev_test")
    v_ht = get_acc(veri, "dev_test")
    if n_ht and v_ht:
        delta_ht = v_ht - n_ht
        print(f"  {'Held-out Accuracy':<25} {n_ht:>10.4f} {v_ht:>10.4f} {delta_ht:>+10.4f}")

    # Per-label
    labels_of_interest = ["SUPPORTS", "REFUTES", "NOT ENOUGH INFO"]
    n_pl = get_per_label(neural, "dev")
    v_pl = get_per_label(veri, "dev")
    print(f"\n  {'Per-class (dev)':<25} {'Neural':>10} {'VERI v2':>10} {'Delta':>10}")
    print(f"  {'-'*55}")
    for lbl in labels_of_interest:
        na = n_pl.get(lbl, {}).get("accuracy", 0)
        va = v_pl.get(lbl, {}).get("accuracy", 0)
        d = va - na
        marker = " ← KEY" if lbl == "REFUTES" else ""
        print(f"  {lbl:<25} {na:>10.4f} {va:>10.4f} {d:>+10.4f}{marker}")

    # Calibration
    n_ece = neural.get("dev", {}).get("ece", "N/A")
    v_ece = veri.get("dev", {}).get("ece", "N/A")
    print(f"\n  {'ECE':<25} {n_ece:>10.4f} {v_ece:>10.4f}")

    # Verdict
    print(f"\n  {'='*55}")
    if delta_dev > 0.005:
        print(f"  ✓ NST-VERI v2 WINS on dev by {delta_dev:+.4f}")
    elif delta_dev < -0.005:
        print(f"  ✗ Neural baseline wins on dev by {-delta_dev:+.4f}")
    else:
        print(f"  ~ Results within noise margin ({delta_dev:+.4f})")

    v_refutes = v_pl.get("REFUTES", {}).get("accuracy", 0)
    n_refutes = n_pl.get("REFUTES", {}).get("accuracy", 0)
    if v_refutes > n_refutes:
        print(f"  ✓ REFUTES bottleneck improved: {n_refutes:.4f} → {v_refutes:.4f}")
    else:
        print(f"  ✗ REFUTES still worse: {n_refutes:.4f} → {v_refutes:.4f}")
    print(f"  {'='*55}")

In [ ]:
# ============================================================
# Cell 12: FULL-DATA NST-VERI v2 (only after 10K shows signal)
# ============================================================
import subprocess, sys, json, os, shutil, time

outdir = "/content/nst/outputs_veri_v2_full"
if os.path.exists(outdir):
    shutil.rmtree(outdir)

print("=" * 65)
print("  FULL-DATA NST-VERI v2 — DeBERTa-v3-large + LoRA")
print("=" * 65)
t0 = time.time()

result = subprocess.run(
    [sys.executable, "main.py", "train-fever-veri-v2",
     "--config", "configs/fever_veri_v2_full_a100.yaml",
     "--outdir", outdir],
    cwd="/content/nst",
    capture_output=False, text=True,
    timeout=14400,  # 4 hour timeout
)

elapsed = time.time() - t0
print(f"\n  Full-data v2 done in {elapsed:.0f}s (exit={result.returncode})")

report_path = os.path.join(outdir, "report.json")
if os.path.exists(report_path):
    report = json.load(open(report_path))
    print(f"\n  Best dev accuracy:  {report.get('best_dev_acc', 'N/A')}")
    print(f"  Temperature:        {report.get('temperature', 'N/A')}")

    dev = report.get('dev', {})
    print(f"\n  Final dev accuracy: {dev.get('accuracy', 'N/A')}")
    for lbl, stats in dev.get('per_label', {}).items():
        print(f"    {lbl}: {stats.get('accuracy', 'N/A')} (n={stats.get('count', '?')})")

    dev_test = report.get('dev_test')
    if dev_test:
        print(f"\n  Held-out: {dev_test.get('accuracy', 'N/A')}")
        for lbl, stats in dev_test.get('per_label', {}).items():
            print(f"    {lbl}: {stats.get('accuracy', 'N/A')}")

    json.dump(report, open("/content/nst/results_veri_v2_full.json", "w"), indent=2)
else:
    print("  WARNING: No report.json found")

In [ ]:
# ============================================================
# Cell 13: NST v3 — Focal Loss + R-Drop + Symbolic Boost
# ============================================================
import subprocess, sys, json, os, shutil, time

# Pull latest code
subprocess.run(['git', 'pull', 'origin', 'main'], cwd='/content/nst', check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.'], cwd='/content/nst',
               capture_output=True)

outdir = '/content/nst/outputs_v3_10k'
if os.path.exists(outdir):
    shutil.rmtree(outdir)

print('=' * 65)
print('  NST v3 — Focal Loss + R-Drop + Symbolic Boost (10K)')
print('=' * 65)
t0 = time.time()

result = subprocess.run(
    [sys.executable, 'main.py', 'train-fever-v3',
     '--config', 'configs/fever_v3_10k_a100.yaml',
     '--outdir', outdir],
    cwd='/content/nst',
    capture_output=False, text=True,
    timeout=7200,
)

elapsed = time.time() - t0
print(f'\n  V3 done in {elapsed:.0f}s (exit={result.returncode})')

report_path = os.path.join(outdir, 'report.json')
if os.path.exists(report_path):
    report = json.load(open(report_path))
    print(f"\n  Best dev accuracy:  {report.get('best_dev_acc', 'N/A')}")

    dev = report.get('dev', {})
    print(f"\n  Final dev accuracy: {dev.get('accuracy', 'N/A')}")
    print(f"  ECE: {dev.get('ece', 'N/A')}")
    for lbl, stats in dev.get('per_label', {}).items():
        print(f"    {lbl}: {stats.get('accuracy', 'N/A')} (n={stats.get('count', '?')})")

    dev_test = report.get('dev_test')
    if dev_test:
        print(f"\n  Held-out: {dev_test.get('accuracy', 'N/A')}")
        for lbl, stats in dev_test.get('per_label', {}).items():
            print(f"    {lbl}: {stats.get('accuracy', 'N/A')}")

    # Compare with neural baseline
    baseline_path = '/content/nst/results_neural_10k.json'
    if os.path.exists(baseline_path):
        bl = json.load(open(baseline_path))
        bl_dev = bl.get('dev', {})
        bl_dt = bl.get('dev_test', {})
        print(f'\n{"="*65}')
        print('  V3 vs Neural Baseline')
        print(f'{"="*65}')
        print(f"  Dev acc:     {dev.get('accuracy','?')} vs {bl_dev.get('accuracy','?')}  "
              f"(delta={float(dev.get('accuracy',0))-float(bl_dev.get('accuracy',0)):+.4f})")
        if dev_test and bl_dt:
            print(f"  Held-out:    {dev_test.get('accuracy','?')} vs {bl_dt.get('accuracy','?')}  "
                  f"(delta={float(dev_test.get('accuracy',0))-float(bl_dt.get('accuracy',0)):+.4f})")
        for lbl in ['REFUTES','SUPPORTS','NOT ENOUGH INFO']:
            v3_a = dev.get('per_label',{}).get(lbl,{}).get('accuracy','?')
            bl_a = bl_dev.get('per_label',{}).get(lbl,{}).get('accuracy','?')
            if v3_a != '?' and bl_a != '?':
                print(f"  {lbl:20s}: {v3_a} vs {bl_a}  (delta={float(v3_a)-float(bl_a):+.4f})")

    json.dump(report, open('/content/nst/results_v3_10k.json', 'w'), indent=2)
else:
    print('  WARNING: No report.json found')
